# 8. Aspekt środowiskowy — Green IT

**Autor:** Marta Gałuszka · **2026-08** · projekt dyplomowy *Smart Energy Model*

> **Green IT w projekcie dyplomowym** — każdy projekt powinien uzasadnić **świadome wybory** pod kątem kosztu obliczeniowego, nie tylko metryk ML.

Ten notebook uzupełnia slajd 10 w [`03_prezentacja_dyplomowa.ipynb`](03_prezentacja_dyplomowa.ipynb) i sekcję Green IT w [`05_raport_wynikow.ipynb`](05_raport_wynikow.ipynb).

| Notebook | Rola |
|----------|------|
| [`02_ML_predykcja_PV.ipynb`](02_ML_predykcja_PV.ipynb) | pełny research (Ridge / RF / XGB) |
| [`03_prezentacja_dyplomowa.ipynb`](03_prezentacja_dyplomowa.ipynb) | slajd 10 — narracja obrony |
| **Ten plik (`06`)** | tabela efektywności + dokumentacja wyborów |
| [`05_raport_wynikow.ipynb`](05_raport_wynikow.ipynb) | raport Markdown z CSV |

**Regresja PV** — zamiast *accuracy* używamy **Test MAE**, **R²**, **gap train→test** (przeuczenie) oraz kosztu: czas `.fit()`, rozmiar `.joblib`.


In [1]:
# 0) Setup
from pathlib import Path
import os
import sys
import time
import tempfile
import importlib.util

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

plt.rcParams['figure.figsize'] = (10, 4)
%matplotlib inline

def _find_root() -> Path:
    here = Path.cwd().resolve()
    for cand in (here, here.parent):
        if (cand / 'src').is_dir() and (cand / 'docs').is_dir():
            return cand
    return here.parent if here.name == 'notebooks' else here

ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

DATA = ROOT / 'data' / 'processed'
FIGURES = ROOT / 'reports' / 'figures'
MODELS = ROOT / 'models'
REPORTS = ROOT / 'reports'

def show_table(df):
    d = df.copy()
    for c in d.columns:
        if pd.api.types.is_float_dtype(d[c]):
            d[c] = d[c].map(lambda x: f'{x:.3f}' if pd.notna(x) else '')
    cols = [str(c) for c in d.columns]
    lines = ['| ' + ' | '.join(cols) + ' |', '| ' + ' | '.join(['---'] * len(cols)) + ' |']
    for _, row in d.iterrows():
        lines.append('| ' + ' | '.join(str(v) for v in row.tolist()) + ' |')
    display(Markdown('\n'.join(lines)))

print('ROOT:', ROOT)


ROOT: /path/to/smart-energy-model


## 1. Porównanie efektywności modeli

**Źródło metryk jakości:** `data/processed/hourly_algorithm_comparison.csv` (ten sam split 80/20 po dniach co [`compare_algorithms_hourly.py`](../scripts/analysis/compare_algorithms_hourly.py)).

**Koszt obliczeniowy** mierzymy lokalnie (jednorazowy benchmark `.fit()` + rozmiar artefaktu `.joblib`).

**Efficiency (regresja):**

$$\text{efficiency} = \frac{1 / \text{Test MAE}}{\text{czas fit [s]} \times \text{rozmiar [MB]}}$$

Im wyżej, tym lepszy stosunek jakości prognozy do zużycia CPU/RAM/dysku. (Analogicznie do szablonu kursu: *accuracy / (czas × pamięć)*, ale dla regresji PV.)


In [2]:
# 1) Metryki jakości z CSV (offline, powtarzalne)
cmp = pd.read_csv(DATA / 'hourly_algorithm_comparison.csv')
quality = cmp[['label', 'test_mae_hour', 'test_r2_hour', 'gap_hour', 'daily_mae', 'verdict']].copy()

# 2) Benchmark czasu i rozmiaru (lokalnie, ten sam protokół co skrypt)
from dotenv import load_dotenv
load_dotenv(ROOT / '.env')
_db = os.getenv('DATABASE_PATH', 'data/energy_model.db')
if not os.path.isabs(_db):
    os.environ['DATABASE_PATH'] = str(ROOT / _db)

from src.features.pv_features_hourly_extended import (
    HOURLY_FEATURE_COLUMNS_PRODUCTION,
    TARGET_COLUMN,
    load_hourly_training_frame_extended,
)

spec = importlib.util.spec_from_file_location(
    'compare_hourly', ROOT / 'scripts/analysis/compare_algorithms_hourly.py'
)
compare_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(compare_mod)

df = load_hourly_training_frame_extended('2025-06-01', '2026-05-31')
days = df['day'].unique()
train_days, test_days = train_test_split(days, test_size=0.2, random_state=42)
train = df[df['day'].isin(train_days)]
test = df[df['day'].isin(test_days)]
X_train = train[HOURLY_FEATURE_COLUMNS_PRODUCTION]
y_train = train[TARGET_COLUMN]
X_test = test[HOURLY_FEATURE_COLUMNS_PRODUCTION]
y_test = test[TARGET_COLUMN]

bench_rows = []
for key, pipeline in compare_mod._get_models().items():
    import copy
    model = copy.deepcopy(pipeline)
    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    fit_s = time.perf_counter() - t0
    tmp = Path(tempfile.gettempdir()) / f'green_it_{key}.joblib'
    joblib.dump(model, tmp)
    size_mb = max(tmp.stat().st_size / (1024 ** 2), 0.01)
    tmp.unlink(missing_ok=True)
    bench_rows.append({
        'label': compare_mod.MODEL_LABELS[key],
        'fit_s': max(round(fit_s, 2), 0.01),
        'artifact_mb': round(size_mb, 2),
    })

bench = pd.DataFrame(bench_rows)
green = quality.merge(bench, on='label', how='left')
green['quality'] = 1 / green['test_mae_hour']
green['efficiency'] = green['quality'] / (green['fit_s'] * green['artifact_mb'])

display_cols = [
    'label', 'test_mae_hour', 'test_r2_hour', 'gap_hour',
    'fit_s', 'artifact_mb', 'efficiency', 'verdict',
]
show_table(green[display_cols].round(3))

prod_path = MODELS / 'pv_hourly_model.joblib'
if prod_path.exists():
    prod_mb = prod_path.stat().st_size / (1024 ** 2)
    print(f"\nModel produkcyjny: {prod_path.name} → {prod_mb:.2f} MB")


FoxESS-Cloud Open API version 2.9.15


/path/to/smart-energy-model/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


✓ Target godzinowy = PVEnergyTotal (Δ licznika, jak w app): 67 dni, 916 godzin
✓ Dodano flagi śniegu z modelu topnienia (dni ze śniegiem: 0 / 67)


✓ Dodano flagę mgły (dni z mgłą: 0 / 67)
📊 Statystyki godzin produkcji:
   Najwcześniejsza: 5:00
   Najpóźniejsza: 20:00
   Średni wschód słońca: 4.73
   Średni zachód słońca: 20.71


/path/to/smart-energy-model/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/path/to/smart-energy-model/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/path/to/smart-energy-model/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


| label | test_mae_hour | test_r2_hour | gap_hour | fit_s | artifact_mb | efficiency | verdict |
| --- | --- | --- | --- | --- | --- | --- | --- |
| Ridge | 0.831 | 0.526 | -0.003 | 0.010 | 0.010 | 12031.294 | ✅ Nie przeuczony |
| RF (prod.) | 0.602 | 0.675 | 0.096 | 0.110 | 0.510 | 29.613 | ✅ Nie przeuczony |
| XGBoost | 0.614 | 0.654 | 0.470 | 0.260 | 0.660 | 9.494 | ❌ Przeuczony |


Model produkcyjny: pv_hourly_model.joblib → 1.31 MB


In [3]:
fig, ax = plt.subplots(figsize=(8, 5))
for _, row in green.iterrows():
    ax.scatter(row['fit_s'], row['test_mae_hour'], s=row['artifact_mb'] * 400 + 80, alpha=0.75)
    ax.annotate(row['label'], (row['fit_s'], row['test_mae_hour']), xytext=(6, 4), textcoords='offset points')
ax.set_xlabel('Czas .fit() [s]')
ax.set_ylabel('Test MAE [kWh/h]')
ax.set_title('Green IT — jakość vs koszt treningu (wielkość bąbla ≈ rozmiar .joblib)')
ax.grid(True, linestyle='--', alpha=0.4)
out = FIGURES / 'green_it_efficiency_scatter.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print('Zapisano:', out.relative_to(ROOT))


Zapisano: reports/figures/green_it_efficiency_scatter.png


/var/folders/w7/4rpbx0_d3lsfbxxbc9b6t4fh0000gn/T/ipykernel_55165/2027862860.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Świadomy wybór: czy +2% jakości warte 10× dłuższego treningu?

W klasyfikacji często porównuje się *accuracy* 93% vs 95%. Tutaj **regresja godzinowa PV**:

| Pytanie | Ridge | RF (prod.) | XGBoost |
|---------|-------|------------|---------|
| Test MAE | ~0,83 | **~0,60** | ~0,61 |
| Gap (overfit) | ~0 | **~0,10** | **~0,47** ❌ |
| Werdykt | baseline tani | **wybrany na produkcję** | odrzucony mimo niższego train MAE |

**Wniosek:** XGBoost nie daje istotnie lepszego Test MAE niż RF (~0,01 kWh/h), ale trenuje **~2–3× dłużej**, ma **większy artefakt** i **4× większy gap** → więcej ryzyka przeuczenia i kosztowniejszy tuning. Wybrałam **Random Forest 200 drzew × 16 cech** — kompromis MAE, stabilności i kosztu inferencji (por. [`docs/CHANGELOG_ML.md`](../docs/CHANGELOG_ML.md)).

Analogicznie do przykładu z kursu (*LogReg 93% vs GB 95%*): **nie płacimy 10× kosztem za marginalny zysk**, gdy RF już spełnia wymagania operacyjne (MAPE live ~8% w erze dual).


## 3. Optymalizacje w kodzie

| Optymalizacja | Gdzie | Po co |
|---------------|-------|-------|
| **`n_jobs=-1`** | RF / XGB w pipeline | pełne wykorzystanie rdzeni CPU przy treningu porównawczym |
| **16 cech zamiast 19** | `HOURLY_FEATURE_COLUMNS_PRODUCTION` | mniejszy wektor, ten sam wynik — ablacja w [`ablation_study.py`](../scripts/analysis/ablation_study.py) |
| **Jeden model pogodowy (ICON)** | loader cech | bez ensemble wielu API → mniej requestów i energii sieciowej |
| **Retrening tygodniowy, nie ciągły** | `launchd` niedziela 04:30 | [`train_dual_weekly.sh`](../mlops/train_dual_weekly.sh) zamiast cloud GPU 24/7 |
| **Prognoza bez domyślnego retreningu** | `mlops/forecast_pv.py` | `--retrain` tylko gdy brak `.joblib` |
| **Pin zależności** | `requirements.txt` | reprodukowalność, brak „przypadkowych” upgrade'ów bibliotek |
| **EDA na agregatach dziennych** | notebook `01` | wykresy z `resample('D')`, nie pełny dump 5-minutowych szeregów w pamięci |

> **Early stopping:** w tym projekcie RF nie wymaga iteracji epok; dla XGBoost shadow używamy umiarkowanego `n_estimators=200` zamiast długiego grid search na produkcji.


In [4]:
from src.models.pv_hourly_predictor import (
    RF_N_ESTIMATORS, RF_MAX_DEPTH, RF_MIN_SAMPLES_LEAF,
)
from sklearn.ensemble import RandomForestRegressor

rf_snippet = RandomForestRegressor(
    n_estimators=RF_N_ESTIMATORS,
    max_depth=RF_MAX_DEPTH,
    min_samples_leaf=RF_MIN_SAMPLES_LEAF,
    n_jobs=-1,
    random_state=42,
)
print('RF produkcyjny:', rf_snippet)
print('n_estimators =', RF_N_ESTIMATORS, '| max_depth =', RF_MAX_DEPTH)


RF produkcyjny: RandomForestRegressor(max_depth=6, min_samples_leaf=20, n_estimators=200,
                      n_jobs=-1, random_state=42)
n_estimators = 200 | max_depth = 6


## 4. Docker slim

Obraz API używa **`python:3.11-slim`** zamiast pełnego `python:3.11` — mniejszy footprint kontenera (mniej warstw systemowych, szybszy pull/deploy).

```dockerfile
FROM python:3.11-slim
...
RUN pip install --no-cache-dir -r requirements.txt
```

Dodatkowo: `--no-install-recommends` przy `apt-get`, kopiowanie tylko `api/`, `src/`, `config/` (bez notebooków i raw data).


In [5]:
dockerfile = (ROOT / 'Dockerfile').read_text(encoding='utf-8')
print(dockerfile.split('\n')[0])
print('...')
for line in dockerfile.split('\n')[8:11]:
    print(line)


FROM python:3.11-slim
...
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt



## 5. Cloud vs lokalny MLOps (OZE)

**Decyzja projektu:** pipeline produkcyjny działa na **Macu użytkownika** (`launchd`), nie na stałej maszynie w chmurze.

| Aspekt | Wybór | Uzasadnienie Green IT |
|--------|-------|------------------------|
| Trening | host lokalny, 1×/tydzień | brak 24/7 GPU/VM w cloud |
| Inferencja | FastAPI lokalnie / Docker | prognoza dzienna ~ms, bez round-trip do regionu chmurowego |
| Sync danych | FoxESS + Open-Meteo | tylko niezbędne API, cache w SQLite |
| Gdyby cloud | region z OZE | np. GCP `europe-west4` / Azure Nordics — gdyby deploy produkcyjny był w chmurze |

Energia **fizyczna** (PV + bateria) jest celem aplikacji; warstwa ML nie powinna zużywać więcej energii niż potrzeba do codziennej prognozy.


## 6. Dokumentacja wyborów (podsumowanie)

| Decyzja | Alternatywa | Dlaczego tak |
|---------|-------------|--------------|
| **RF 16 cech** | XGBoost, 19 cech | podobny Test MAE, **4× mniejszy gap**, mniejszy tuning |
| **Ridge** | — | baseline ~0,01 s treningu — punkt odniesienia kosztu |
| **Target PVE (Δ licznika)** | surowa moc | zgodność z aplikacją FoxESS, bez post-processingu skali |
| **Tygodniowy retrain** | ciągły retrening | [`config/launchd/`](../config/launchd/) — przewidywalny koszt |
| **python:3.11-slim** | pełny obraz | mniejszy kontener API |
| **Brak MLflow w prod** | centralny tracking server | prostszy stack, mniej moving parts na obronę |

**Powiązane materiały:**
- Slajd 10: [`03_prezentacja_dyplomowa.ipynb`](03_prezentacja_dyplomowa.ipynb)
- Historia decyzji: [`docs/CHANGELOG_ML.md`](../docs/CHANGELOG_ML.md)
- Raport: [`reports/model_comparison.md`](../reports/model_comparison.md)
- Checklist dyplom: [`docs/PLAN_DYPLOM_CHECKLIST.md`](../docs/PLAN_DYPLOM_CHECKLIST.md) § Green IT


In [6]:
summary_lines = [
    '# Green IT — Smart Energy Model',
    '',
    f'*Wygenerowano: {pd.Timestamp.now():%Y-%m-%d %H:%M} z `06_aspekt_srodowiskowy.ipynb`*',
    '',
    '## Tabela efektywności',
    '',
]
show_df = green[display_cols].round(3)
cols = list(show_df.columns)
summary_lines.append('| ' + ' | '.join(cols) + ' |')
summary_lines.append('| ' + ' | '.join(['---'] * len(cols)) + ' |')
for _, row in show_df.iterrows():
    summary_lines.append('| ' + ' | '.join(str(v) for v in row.tolist()) + ' |')

summary_lines += [
    '',
    '## Werdykt',
    '- Produkcja: **Random Forest 16 cech** — najlepszy kompromis Test MAE, gap i koszt.',
    '- XGBoost odrzucony mimo niskiego train MAE (duży gap).',
    '- Docker: `python:3.11-slim`; MLOps: launchd tygodniowy retrening na hoście.',
    '',
]
out_md = REPORTS / 'green_it_summary.md'
out_md.write_text('\n'.join(summary_lines) + '\n', encoding='utf-8')
print('Zapisano:', out_md.relative_to(ROOT))


Zapisano: reports/green_it_summary.md
